# einops-repeat-broadcast — ex3: few-shot prototype broadcast for cosine-similarity classifier

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat-broadcast`. Running the final beacon cell reports progress against the `Einops: Repeat-as-broadcast` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.repeat as broadcast — quick refresher

`einops.repeat(x, 'a b -> a n b', n=N)` inserts a NEW axis of length `N` *without* allocating `N` copies — internally the new axis is a stride-0 view. This lets you 'pair every X with every Y' (broadcast to `(NX, NY, ...)`) without quadratic memory. The downstream elementwise op materialises the result lazily.

### Exercise 3 — few-shot prototype broadcast for cosine-similarity classifier

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose two `einops.repeat` broadcasts (queries vs prototypes) to produce an `(N, C)` cosine-similarity logit matrix without materialising the `(N, C, D)` intermediate.
> Keywords: few-shot, prototype, cosine-similarity, metric-learning, integrative
> ```

**KCs targeted:** `repeat-inserts-zero-stride-axis`, `repeat-pair-every-with-every`

ex1 paired rays × triangles. ex2 broadcast a per-token bias. This one is the *metric-learning* facet: pair every query with every class prototype to get a cosine-similarity logit matrix — the prototypical-net forward pass.

Implement `ex3_proto_logits(queries, prototypes)`:

1. `queries` has shape `(N, D)` — N query embeddings.
2. `prototypes` has shape `(C, D)` — one mean-embedding per class.
3. Use `einops.repeat` to expand:
   - `queries`  → `(N, C, D)` with `'n d -> n c d'`, `c=C`
   - `prototypes` → `(N, C, D)` with `'c d -> n c d'`, `n=N`
4. Compute cosine similarity element-wise along `D`:
   `cos(a, b) = (a · b) / (||a|| ||b||)`
5. Return the `(N, C)` similarity matrix.

Output dtype: `float32`. Values in `[-1, 1]`. The dot product collapses the `D` axis — the broadcast machinery supplies the `(N, C)` shape.

In [ ]:
def ex3_proto_logits(queries: Tensor, prototypes: Tensor) -> Tensor:
    """Return (N, C) cosine-similarity logits via einops.repeat broadcast."""
    raise NotImplementedError()


def _test_ex3():
    # Hand-checked case: orthonormal-ish prototypes.
    queries = t.tensor([
        [1.0, 0.0, 0.0],   # exactly the x-axis
        [0.0, 1.0, 0.0],   # exactly the y-axis
        [0.7071, 0.7071, 0.0],  # 45° between x and y
    ])
    prototypes = t.tensor([
        [1.0, 0.0, 0.0],   # class 0: x-axis
        [0.0, 1.0, 0.0],   # class 1: y-axis
        [0.0, 0.0, 1.0],   # class 2: z-axis
    ])
    logits = ex3_proto_logits(queries, prototypes)
    assert logits.shape == (3, 3), f'expected (3,3), got {tuple(logits.shape)}'
    assert logits.dtype == t.float32, f'expected float32, got {logits.dtype}'
    # Diagonal: query == prototype → cos = 1 for rows 0 and 1.
    assert abs(logits[0, 0].item() - 1.0) < 1e-4
    assert abs(logits[1, 1].item() - 1.0) < 1e-4
    # Off-diagonal of orthogonal pairs: 0.
    assert abs(logits[0, 1].item()) < 1e-4
    assert abs(logits[0, 2].item()) < 1e-4
    # 45° query gives 1/sqrt(2) to both class 0 and class 1, 0 to class 2.
    assert abs(logits[2, 0].item() - 0.7071) < 1e-3
    assert abs(logits[2, 1].item() - 0.7071) < 1e-3
    assert abs(logits[2, 2].item()) < 1e-3
    # Argmax-picks-correct on diagonal.
    assert logits.argmax(dim=1).tolist()[:2] == [0, 1]

    # Bounds check on random embeddings.
    rng = t.Generator().manual_seed(42)
    Q = t.randn(20, 64, generator=rng)
    P = t.randn(5, 64, generator=rng)
    L = ex3_proto_logits(Q, P)
    assert L.shape == (20, 5)
    assert L.min().item() >= -1.0 - 1e-5
    assert L.max().item() <=  1.0 + 1e-5

    # Symmetric: ex3(q, p)[i, j] == ex3(p, q)[j, i].
    L_t = ex3_proto_logits(P, Q)
    assert t.allclose(L, L_t.T, atol=1e-5)

    # --- Visualization: heatmap of class logits for a synthetic few-shot run ---
    n_classes = 4
    centres = t.randn(n_classes, 8, generator=rng)
    # 12 queries: 3 near each class centre + light noise.
    query_rows = []
    for c in range(n_classes):
        for _ in range(3):
            query_rows.append(centres[c] + 0.3 * t.randn(8, generator=rng))
    queries_v = t.stack(query_rows)  # (12, 8)
    L_v = ex3_proto_logits(queries_v, centres)
    fig, ax = plt.subplots(figsize=(4, 5))
    im = ax.imshow(L_v.numpy(), cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xlabel('class prototype')
    ax.set_ylabel('query index')
    ax.set_title('ex3 cosine-similarity logits')
    plt.colorbar(im, ax=ax, label='cosine sim')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_proto_logits(queries: Tensor, prototypes: Tensor) -> Tensor:
    N, D = queries.shape
    C, _ = prototypes.shape
    q = repeat(queries,    'n d -> n c d', c=C)   # (N, C, D), stride-0 along c
    p = repeat(prototypes, 'c d -> n c d', n=N)   # (N, C, D), stride-0 along n
    dots = (q * p).sum(dim=-1)                    # (N, C)
    q_norm = q.norm(dim=-1)                       # (N, C)
    p_norm = p.norm(dim=-1)                       # (N, C)
    return (dots / (q_norm * p_norm + 1e-12)).to(t.float32)
```

**Two `repeat`s collapse to one shape.** Each query needs a copy per class; each prototype needs a copy per query. `einops.repeat` inserts the missing axis as a stride-0 view, so the `(N, C, D)` intermediate is virtual — no `N*C*D` memory cost until the elementwise op fires.

**Why we don't pre-normalise.** You CAN normalise `queries` and `prototypes` to unit length first and skip the division, and that's what real ProtoNet code does. Here we compute the full formula to make the broadcast pattern visible — the norms are taken AFTER the repeat, on the shared `(N, C, D)` shape, so they too rely on the broadcast.

**Contrast with ex1.** ex1 used `repeat` to pair rays × triangles in a geometry context. This drill is the same broadcast pattern in a metric-learning context — the atom (zero-stride pairing) generalises across domains.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()